### **Multi Head Attention Weight Splits**

#### ***Step_1: Input Embeddings***

In [119]:
import torch
inputs = torch.tensor([
    [0.20, 0.50, 0.10, 0.30, 0.40, 0.60],  # The
    [0.80, 0.70, 0.20, 0.90, 0.30, 0.50],  # cat
    [0.10, 0.30, 0.80, 0.20, 0.60, 0.40],  # quietly
    [0.70, 0.40, 0.90, 0.60, 0.80, 0.30],  # watches
    [0.60, 0.90, 0.70, 0.50, 0.40, 0.80]   # birds
])
inputs.shape

torch.Size([5, 6])

#### ***Step_2:Decide Dimensional Out and Number of Heads***

In [120]:
d_in = inputs.shape[-1]
d_in

6

In [121]:
## d_in = d_out
d_out = d_in
d_out

6

In [122]:
## let's take 3 heads
num_heads = 3

###Head Dimension
head_dim = d_out/num_heads

head_dim

2.0

#### ***Step_3: Trainable Weight Matrix for Key,Query,and Value.***


In [123]:
import torch.nn as nn
weight_Key = nn.Linear(d_in,d_out,bias = False)
weight_query = nn.Linear(d_in,d_out,bias=False)
weight_value = nn.Linear(d_in,d_out,bias=False)

print("Key:",weight_Key.weight)

Key: Parameter containing:
tensor([[ 0.2021, -0.0017,  0.2898, -0.2092,  0.2104, -0.0379],
        [-0.0710,  0.0478, -0.3128,  0.0472,  0.1373,  0.3490],
        [-0.1271,  0.1470,  0.4080, -0.1751,  0.3881, -0.2027],
        [ 0.1800,  0.1599,  0.1140,  0.3228, -0.1650,  0.1073],
        [ 0.0023, -0.3071, -0.0991, -0.2726,  0.1805,  0.0367],
        [ 0.0400, -0.1239,  0.0020, -0.1270,  0.1173,  0.3965]],
       requires_grad=True)


#### ***Step_4: Calculate Key,Query, and Value Matrix***

In [124]:
Queries = inputs @ weight_query.weight
print(Queries)
print("Shape:",Queries.shape)

tensor([[-0.2910, -0.0225, -0.2131,  0.0581,  0.2558,  0.3184],
        [-0.5064,  0.0341, -0.1444,  0.4001,  0.1141,  0.4444],
        [-0.2843,  0.1224, -0.4097,  0.0956,  0.2361,  0.5271],
        [-0.3332,  0.2188, -0.4209,  0.2496,  0.0654,  0.5706],
        [-0.6401, -0.1110, -0.5279,  0.3609,  0.3317,  0.6459]],
       grad_fn=<MmBackward0>)
Shape: torch.Size([5, 6])


In [125]:
Keys = inputs @ weight_Key.weight
print(Keys)
print("Shape:",Keys.shape)

tensor([[ 0.0711, -0.1109, -0.0619, -0.1242,  0.2426,  0.4314],
        [ 0.2692,  0.0514,  0.1684, -0.0241,  0.3063,  0.4793],
        [-0.0494, -0.0700,  0.2257, -0.2967,  0.4949,  0.1408],
        [ 0.1205, -0.0367,  0.4347, -0.3477,  0.6321,  0.1434],
        [ 0.0912,  0.0029,  0.1970, -0.2549,  0.6050,  0.5350]],
       grad_fn=<MmBackward0>)
Shape: torch.Size([5, 6])


In [126]:
Values = inputs @ weight_value.weight
print(Values)
print("Shape:",Values.shape)

tensor([[-0.1979,  0.1474,  0.2331, -0.4320, -0.1433, -0.2452],
        [ 0.0066,  0.2930,  0.3261, -0.7056, -0.3172, -0.5366],
        [-0.1158,  0.3326, -0.0924, -0.3986,  0.0019, -0.2538],
        [-0.0520,  0.4389, -0.0032, -0.6355, -0.1807, -0.3980],
        [-0.1629,  0.4464,  0.2712, -0.8941, -0.2826, -0.6654]],
       grad_fn=<MmBackward0>)
Shape: torch.Size([5, 6])


#### ***Step_5: Unroll the out Dimensions into number of heads and head dimensions.***

In [127]:
num_heads

3

In [128]:
head_dim = int(head_dim)
head_dim

2

In [129]:
num_tokens = inputs.shape[0]
num_tokens

5

In [130]:
##d_out = num_heads * head_dim

In [131]:
Keys = Keys.view(num_tokens,num_heads,head_dim)
Queries = Queries.view(num_tokens,num_heads,head_dim)
Values = Values.view(num_tokens,num_heads,head_dim)
print(Keys)
print("Shape:",Keys.shape)

tensor([[[ 0.0711, -0.1109],
         [-0.0619, -0.1242],
         [ 0.2426,  0.4314]],

        [[ 0.2692,  0.0514],
         [ 0.1684, -0.0241],
         [ 0.3063,  0.4793]],

        [[-0.0494, -0.0700],
         [ 0.2257, -0.2967],
         [ 0.4949,  0.1408]],

        [[ 0.1205, -0.0367],
         [ 0.4347, -0.3477],
         [ 0.6321,  0.1434]],

        [[ 0.0912,  0.0029],
         [ 0.1970, -0.2549],
         [ 0.6050,  0.5350]]], grad_fn=<ViewBackward0>)
Shape: torch.Size([5, 3, 2])


#### ***Step_6: Group Matrices by Number of Heads***

In [132]:
Keys = Keys.view(num_heads,num_tokens,head_dim)
Queries = Queries.view(num_heads,num_tokens,head_dim)
Values = Values.view(num_heads,num_tokens,head_dim)
print(Keys)
print("Shape:",Keys.shape)

tensor([[[ 0.0711, -0.1109],
         [-0.0619, -0.1242],
         [ 0.2426,  0.4314],
         [ 0.2692,  0.0514],
         [ 0.1684, -0.0241]],

        [[ 0.3063,  0.4793],
         [-0.0494, -0.0700],
         [ 0.2257, -0.2967],
         [ 0.4949,  0.1408],
         [ 0.1205, -0.0367]],

        [[ 0.4347, -0.3477],
         [ 0.6321,  0.1434],
         [ 0.0912,  0.0029],
         [ 0.1970, -0.2549],
         [ 0.6050,  0.5350]]], grad_fn=<ViewBackward0>)
Shape: torch.Size([3, 5, 2])


In [133]:
Queries

tensor([[[-0.2910, -0.0225],
         [-0.2131,  0.0581],
         [ 0.2558,  0.3184],
         [-0.5064,  0.0341],
         [-0.1444,  0.4001]],

        [[ 0.1141,  0.4444],
         [-0.2843,  0.1224],
         [-0.4097,  0.0956],
         [ 0.2361,  0.5271],
         [-0.3332,  0.2188]],

        [[-0.4209,  0.2496],
         [ 0.0654,  0.5706],
         [-0.6401, -0.1110],
         [-0.5279,  0.3609],
         [ 0.3317,  0.6459]]], grad_fn=<ViewBackward0>)

#### ***Step_7: Find Attention Scores***

In [134]:
Keys.transpose(1,2)

tensor([[[ 0.0711, -0.0619,  0.2426,  0.2692,  0.1684],
         [-0.1109, -0.1242,  0.4314,  0.0514, -0.0241]],

        [[ 0.3063, -0.0494,  0.2257,  0.4949,  0.1205],
         [ 0.4793, -0.0700, -0.2967,  0.1408, -0.0367]],

        [[ 0.4347,  0.6321,  0.0912,  0.1970,  0.6050],
         [-0.3477,  0.1434,  0.0029, -0.2549,  0.5350]]],
       grad_fn=<TransposeBackward0>)

In [135]:
attention_scores = Queries @ Keys.transpose(1,2)
print(attention_scores)
print("Shape:",attention_scores.shape)

tensor([[[-0.0182,  0.0208, -0.0803, -0.0795, -0.0485],
         [-0.0216,  0.0060, -0.0267, -0.0544, -0.0373],
         [-0.0171, -0.0554,  0.1994,  0.0852,  0.0354],
         [-0.0398,  0.0271, -0.1081, -0.1346, -0.0861],
         [-0.0546, -0.0407,  0.1376, -0.0183, -0.0340]],

        [[ 0.2480, -0.0368, -0.1061,  0.1191, -0.0025],
         [-0.0284,  0.0055, -0.1005, -0.1235, -0.0387],
         [-0.0797,  0.0136, -0.1208, -0.1893, -0.0529],
         [ 0.3250, -0.0486, -0.1031,  0.1911,  0.0091],
         [ 0.0028,  0.0011, -0.1401, -0.1341, -0.0482]],

        [[-0.2697, -0.2303, -0.0377, -0.1465, -0.1211],
         [-0.1700,  0.1231,  0.0076, -0.1326,  0.3449],
         [-0.2396, -0.4205, -0.0587, -0.0978, -0.4466],
         [-0.3549, -0.2819, -0.0471, -0.1960, -0.1262],
         [-0.0803,  0.3023,  0.0322, -0.0993,  0.5463]]],
       grad_fn=<UnsafeViewBackward0>)
Shape: torch.Size([3, 5, 5])


#### ***Find Attention Weights***

In [136]:
context_length = inputs.shape[0]

mask = torch.triu(torch.ones(context_length,context_length),diagonal=1)

attention_scores.masked_fill_(mask.bool(),-torch.inf)

tensor([[[-0.0182,    -inf,    -inf,    -inf,    -inf],
         [-0.0216,  0.0060,    -inf,    -inf,    -inf],
         [-0.0171, -0.0554,  0.1994,    -inf,    -inf],
         [-0.0398,  0.0271, -0.1081, -0.1346,    -inf],
         [-0.0546, -0.0407,  0.1376, -0.0183, -0.0340]],

        [[ 0.2480,    -inf,    -inf,    -inf,    -inf],
         [-0.0284,  0.0055,    -inf,    -inf,    -inf],
         [-0.0797,  0.0136, -0.1208,    -inf,    -inf],
         [ 0.3250, -0.0486, -0.1031,  0.1911,    -inf],
         [ 0.0028,  0.0011, -0.1401, -0.1341, -0.0482]],

        [[-0.2697,    -inf,    -inf,    -inf,    -inf],
         [-0.1700,  0.1231,    -inf,    -inf,    -inf],
         [-0.2396, -0.4205, -0.0587,    -inf,    -inf],
         [-0.3549, -0.2819, -0.0471, -0.1960,    -inf],
         [-0.0803,  0.3023,  0.0322, -0.0993,  0.5463]]],
       grad_fn=<MaskedFillBackward0>)

In [137]:
attention_weights = torch.softmax(attention_scores/head_dim**0.5,dim=-1)
attention_weights

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4951, 0.5049, 0.0000, 0.0000, 0.0000],
         [0.3186, 0.3101, 0.3713, 0.0000, 0.0000],
         [0.2540, 0.2663, 0.2421, 0.2376, 0.0000],
         [0.1924, 0.1943, 0.2205, 0.1975, 0.1953]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4940, 0.5060, 0.0000, 0.0000, 0.0000],
         [0.3290, 0.3514, 0.3196, 0.0000, 0.0000],
         [0.2927, 0.2248, 0.2163, 0.2663, 0.0000],
         [0.2094, 0.2092, 0.1893, 0.1901, 0.2020]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4484, 0.5516, 0.0000, 0.0000, 0.0000],
         [0.3315, 0.2917, 0.3768, 0.0000, 0.0000],
         [0.2265, 0.2385, 0.2816, 0.2534, 0.0000],
         [0.1684, 0.2207, 0.1824, 0.1662, 0.2623]]],
       grad_fn=<SoftmaxBackward0>)

In [138]:
###drop out
drop_out = nn.Dropout(0.0)
attention_weights = drop_out(attention_weights)
attention_weights

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4951, 0.5049, 0.0000, 0.0000, 0.0000],
         [0.3186, 0.3101, 0.3713, 0.0000, 0.0000],
         [0.2540, 0.2663, 0.2421, 0.2376, 0.0000],
         [0.1924, 0.1943, 0.2205, 0.1975, 0.1953]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4940, 0.5060, 0.0000, 0.0000, 0.0000],
         [0.3290, 0.3514, 0.3196, 0.0000, 0.0000],
         [0.2927, 0.2248, 0.2163, 0.2663, 0.0000],
         [0.2094, 0.2092, 0.1893, 0.1901, 0.2020]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4484, 0.5516, 0.0000, 0.0000, 0.0000],
         [0.3315, 0.2917, 0.3768, 0.0000, 0.0000],
         [0.2265, 0.2385, 0.2816, 0.2534, 0.0000],
         [0.1684, 0.2207, 0.1824, 0.1662, 0.2623]]],
       grad_fn=<SoftmaxBackward0>)

#### ***Step_9: Context Vectors***

In [139]:
#context_vector = attention_weights * values

context_vector = attention_weights @ Values
print(context_vector)
print("Shape:",context_vector.shape)

tensor([[[-0.1979,  0.1474],
         [ 0.0197, -0.1451],
         [-0.0440, -0.1781],
         [-0.0213, -0.0674],
         [ 0.0406, -0.1896]],

        [[-0.3172, -0.5366],
         [-0.2153, -0.0968],
         [-0.1746, -0.1871],
         [-0.1383, -0.2361],
         [-0.1183, -0.0779]],

        [[-0.0032, -0.6355],
         [-0.1011, -0.5045],
         [-0.1151, -0.1586],
         [-0.0209, -0.3397],
         [-0.0992, -0.4366]]], grad_fn=<UnsafeViewBackward0>)
Shape: torch.Size([3, 5, 2])


#### ***Step_10: Combine Heads***

In [140]:
### Flatten each token output into each row

context_vector = context_vector.view(num_tokens,num_heads,head_dim)
context_vector #5 Tokens 3 heads, 2 dimensions

tensor([[[-0.1979,  0.1474],
         [ 0.0197, -0.1451],
         [-0.0440, -0.1781]],

        [[-0.0213, -0.0674],
         [ 0.0406, -0.1896],
         [-0.3172, -0.5366]],

        [[-0.2153, -0.0968],
         [-0.1746, -0.1871],
         [-0.1383, -0.2361]],

        [[-0.1183, -0.0779],
         [-0.0032, -0.6355],
         [-0.1011, -0.5045]],

        [[-0.1151, -0.1586],
         [-0.0209, -0.3397],
         [-0.0992, -0.4366]]], grad_fn=<ViewBackward0>)

In [141]:
### d_out = num_heads*head_dim
context_vector = context_vector.contiguous().view(num_tokens,d_out)
context_vector  #Finall Context VEctors

tensor([[-0.1979,  0.1474,  0.0197, -0.1451, -0.0440, -0.1781],
        [-0.0213, -0.0674,  0.0406, -0.1896, -0.3172, -0.5366],
        [-0.2153, -0.0968, -0.1746, -0.1871, -0.1383, -0.2361],
        [-0.1183, -0.0779, -0.0032, -0.6355, -0.1011, -0.5045],
        [-0.1151, -0.1586, -0.0209, -0.3397, -0.0992, -0.4366]],
       grad_fn=<ViewBackward0>)

In [142]:
import torch.nn as nn 

class MultiHeadAttention(nn.Module):

    def __init__(self,d_in,d_out,context_length,drop_out,num_heads,qvk_bias = False):
        super().__init__()
        self.d_out = d_out
        self.Keys = nn.Linear(d_in,d_out,bias=qvk_bias)
        self.Queries = nn.Linear(d_in,d_out,bias=qvk_bias)
        self.Values = nn.Linear(d_in,d_out,bias=qvk_bias)
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.drop_out = nn.Dropout(drop_out)
        self.register_buffer('mask',torch.triu(torch.ones(context_length,context_length),diagonal=1))

    def forward(self,x):
        b,num_tokens,d_in = x.shape
        Keys = self.Keys(x)
        Queries = self.Queries(x)
        Values = self.Values(x)

        Keys = Keys.view(b,num_tokens,self.num_heads,self.head_dim)
        Queries = Queries.view(b,num_tokens,self.num_heads,self.head_dim)
        Values = Values.view(b,num_tokens,self.num_heads,self.head_dim)


        Keys = Keys.transpose(1, 2)
        Queries = Queries.transpose(1, 2)
        Values = Values.transpose(1, 2)


        attention_scores = Queries @ Keys.transpose(2,3)

        attention_scores.masked_fill_(self.mask.bool(),-torch.inf)

        attention_weights = torch.softmax(attention_scores/self.head_dim**0.5,dim=-1)

        attention_weights = self.drop_out(attention_weights)

        context_vector = attention_weights @ Values

        context_vector = context_vector.transpose(1, 2)

        context_vector = context_vector.contiguous().view(b,num_tokens,self.d_out)

        return context_vector

        


In [143]:
batch = torch.stack((inputs,inputs),dim=0)

context_length = batch.shape[1]
d_in = batch.shape[-1]
d_out = d_in
d_out

6

In [144]:
torch.manual_seed(123)
mha = MultiHeadAttention(d_in,d_out,context_length,0.0,2)
context_vectors = mha(batch)
print(context_vectors)

tensor([[[-0.3751,  0.1714,  0.1304,  0.1801,  0.1566,  0.0864],
         [-0.4650,  0.2024,  0.1194,  0.2132,  0.2975,  0.0164],
         [-0.3225,  0.2271,  0.0747,  0.1952,  0.3375,  0.0136],
         [-0.2790,  0.2525,  0.0516,  0.1690,  0.4476, -0.0093],
         [-0.3389,  0.2761,  0.0292,  0.2002,  0.4654, -0.0069]],

        [[-0.3751,  0.1714,  0.1304,  0.1801,  0.1566,  0.0864],
         [-0.4650,  0.2024,  0.1194,  0.2132,  0.2975,  0.0164],
         [-0.3225,  0.2271,  0.0747,  0.1952,  0.3375,  0.0136],
         [-0.2790,  0.2525,  0.0516,  0.1690,  0.4476, -0.0093],
         [-0.3389,  0.2761,  0.0292,  0.2002,  0.4654, -0.0069]]],
       grad_fn=<ViewBackward0>)
